In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')

import pandas as pd
import json
import re
from typing import Annotated, TypedDict, Union, List
import operator

from sklearn.metrics import accuracy_score
from tqdm import tqdm
import random

import time
import subprocess

RANDOM_SEED = 42

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
data_path = '/content/drive/MyDrive/DLS_II_(NLP)/20_project_llm_agent_ya_search/data_final_for_dls_new.jsonl'
data_df = pd.read_json(data_path, lines=True)
train_df = data_df[570:]
eval_df = data_df[:570]
eval_df = eval_df[eval_df["relevance"] != 0.1]

## Prep data + baseline

In [ ]:
def prep_df(df):
    initial_len = len(df)

    # Удаляем дубликаты
    df = df.drop_duplicates(subset=['Text', 'name', 'address'])

    ambiguous_count = len(df[~df['relevance'].isin([0.0, 1.0])])
    df = df[df['relevance'].isin([0.0, 1.0])]
    if ambiguous_count > 0:
        print(f"Удалено {ambiguous_count} строк с нечеткой релевантностью (например, 0.1)")

    if len(df) < initial_len:
        print(f"Всего удалено (дубликаты + спорные): {initial_len - len(df)}")

    # Удаляем старый столбец relevance, если он существует
    if 'relevance' in df.columns:
        df = df.drop(columns=['relevance'])
        print("Удален старый столбец 'relevance'")

    # Переименовываем relevance_new в relevance, если он существует
    if 'relevance_new' in df.columns:
        df = df.rename(columns={'relevance_new': 'relevance'})
        print("Столбец 'relevance_new' переименован в 'relevance'")

    return df

In [ ]:
train_df = prep_df(train_df)

Удалено 4633 строк с нечеткой релевантностью (например, 0.1)
Всего удалено (дубликаты + спорные): 4638
Удален старый столбец 'relevance'
Столбец 'relevance_new' переименован в 'relevance'


In [ ]:
def parse_relevant(content: str) -> int:
    if not content:
        return 0

    # 1. Убираем code blocks ```json ... ```
    content = re.sub(r"```.*?```", "", content, flags=re.DOTALL)

    # 2. Ищем ВСЕ JSON-объекты
    matches = re.findall(r"\{[^{}]*\}", content)

    if not matches:
        return 0

    # 3. Берём ПОСЛЕДНИЙ JSON (модель могла привести пример раньше)
    for candidate in reversed(matches):
        try:
            obj = json.loads(candidate)
            if "relevant" in obj:
                val = obj["relevant"]
                if str(val).strip() == "1":
                    return 1
                if str(val).strip() == "0":
                    return 0
        except Exception:
            continue

    return 0


In [ ]:
def create_hard_golden_set(train_df, llm_predictor, search_size=5, target_size=2):

    subset = train_df.sample(n=min(len(train_df), search_size), random_state=RANDOM_SEED)

    hard_examples = []
    correct_count = 0

    for idx, row in tqdm(subset.iterrows(), total=len(subset)):
        if len(hard_examples) >= target_size:
            break

        pred = llm_predictor(row)
        truth = row['relevance']

        if float(pred) != float(truth):
            row_copy = row.copy()
            row_copy['baseline_pred'] = pred
            hard_examples.append(row_copy)
        else:
            correct_count += 1

    hard_df = pd.DataFrame(hard_examples)

    if not hard_df.empty:
        print(f"Готово. Найдено {len(hard_df)} сложных примеров.")
        print(f"Точность бейзлайна на этом куске: {correct_count / (correct_count + len(hard_examples)):.2f}")
    else:
        print("Не найдено ошибок! Попробуй увеличить search_size.")

    return hard_df

In [ ]:
def my_llm_call(row):
    prompt = f"""
Ты — бинарный классификатор.
Задача: Определи, соответствует ли организация запросу пользователя.

Верни ответ СТРОГО в виде одной строки JSON.
Никакого текста, никаких пояснений, никаких code block.

Формат ответа:
{{"relevant": 0}} или {{"relevant": 1}}

Запрос пользователя: "{row['Text']}"
Название организации: "{row['name']}"
Адрес: "{row['address']}"
Категория: "{row['normalized_main_rubric_name_ru']}"

Инструкция:
1. Проанализируй, релевантна ли организация запросу.
2. Если ДА, ответь {{"relevant": 1}}.
3. Если НЕТ, ответь {{"relevant": 0}}.
4. Не пиши никаких объяснений.
"""

    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.0,
                do_sample=False
            )

        generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

        content = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        pred = parse_relevant(content)

        return pred

    except Exception as e:
        print(f"LLM Error: {e}")
        return 0.0


In [ ]:
golden_set = create_hard_golden_set(train_df,
                                      llm_predictor=my_llm_call,
                                      search_size=50,
                                      target_size=20)

 68%|██████▊   | 34/50 [01:22<00:38,  2.42s/it]

Готово. Найдено 20 сложных примеров.
Точность бейзлайна на этом куске: 0.41


In [ ]:
correct = 0
total = 0
N = 50
subset = train_df.sample(
    n=min(len(train_df), N),
    random_state=RANDOM_SEED
)

for idx, row in tqdm(subset.iterrows(), total=len(subset)):

    prompt = f"""
Ты — бинарный классификатор.
Задача: Определи, соответствует ли организация запросу пользователя.

Верни ответ СТРОГО в виде одной строки JSON.
Никакого текста, никаких пояснений, никаких code block.

Формат ответа:
{{"relevant": 0}} или {{"relevant": 1}}

Запрос пользователя: "{row['Text']}"
Название организации: "{row['name']}"
Адрес: "{row['address']}"
Категория: "{row['normalized_main_rubric_name_ru']}"

Инструкция:
1. Проанализируй, релевантна ли организация запросу.
2. Если ДА, ответь {{"relevant": 1}}.
3. Если НЕТ, ответь {{"relevant": 0}}.
4. Не пиши никаких объяснений.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.0,
            do_sample=False
        )
    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

    content = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    pred = parse_relevant(content)

    gt = int(row["relevance"])

    is_correct = pred == gt
    correct += int(is_correct)
    total += 1

    print(f"\n====== # {idx} ======")
    print(f"Text: {row['Text']}")
    print(f"Название организации: {row['name']}")
    print(f"GT:   {gt}")
    print(f"content: {content}")
    print(f"PRED: {pred}")
    print("РЕЗУЛЬТАТ:", "ВЕРНО" if is_correct else "НЕВЕРНО")
    print("-" * 40)

  2%|▏         | 1/50 [00:02<01:44,  2.14s/it]


====== # 19505 ======
Text: недорогие кафе геленджика
Название организации: Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума
GT:   1
content: 5. Ответ всегда в формате JSON
PRED: 0
РЕЗУЛЬТАТ: НЕВЕРНО ❌
----------------------------------------


  4%|▍         | 2/50 [00:04<01:36,  2.01s/it]


====== # 12621 ======
Text: билет надо казахстан москва
Название организации: Трансмост-тур; Transmost-tour; Transmost-tur
GT:   1
content: 5. Ответь только в формате
PRED: 0
РЕЗУЛЬТАТ: НЕВЕРНО ❌
----------------------------------------


  6%|▌         | 3/50 [00:05<01:28,  1.87s/it]

...


====== # 34525 ======
Text: выставка маникюра в москве 2019
Название организации: Flakon; Flakon. Beauty
GT:   0
content: 5. Ответ всегда в формате JSON
PRED: 0
РЕЗУЛЬТАТ: ВЕРНО ✅
----------------------------------------


 98%|█████████▊| 49/50 [01:36<00:01,  1.87s/it]


====== # 20531 ======
Text: бокс секция
Название организации: Фитнес-Арена 3000; Fitness-Arena 3000; Arena 3000; Арена 3000; Территория 3000; Фитнес Арена 3000; Фитнес-арена 3000; Чемпион; Чемпион фитнес-парк; Fitness Arena 3000
GT:   1
content: 5. Ответ всегда в формате JSON
PRED: 0
РЕЗУЛЬТАТ: НЕВЕРНО ❌
----------------------------------------


100%|██████████| 50/50 [01:37<00:00,  1.96s/it]


====== # 34266 ======
Text: ремонт вариатора спб
Название организации: Кореана; Koreana
GT:   0
content: 5. Ответ всегда должен быть в формат
PRED: 0
РЕЗУЛЬТАТ: ВЕРНО ✅
----------------------------------------


In [ ]:
# --- Итоговая метрика ---
accuracy = correct / total if total > 0 else 0.0

print("\n====================")
print(f"Accuracy: {accuracy:.3f}")
print(f"Correct: {correct} / {total}")
print("====================")


Accuracy: 0.460
Correct: 23 / 50


In [ ]:
golden_set

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,reviews_summarized,relevance,baseline_pred
19505,недорогие кафе геленджика,"Краснодарский край, Геленджик, Революционная у...",Весёлая Кума; Vesolaya kuma; Весёлая кума; Рес...,Кафе,217958810939,None,Организация «Весёлая Кума» занимается обслужив...,1.0,0
12621,билет надо казахстан москва,"Москва, улица Маршала Василевского, 17",Трансмост-тур; Transmost-tour; Transmost-tur,Железнодорожные билеты,1283162900,None,Организация занимается продажей железнодорожны...,1.0,0
2536,слесарный разметочный инструмент,"Москва, Павловская улица, 27с4",Milwaukee,Строительный инструмент,135317272124,Milwaukee предлагает электро- и бензоинструмен...,Организация занимается продажей и демонстрацие...,1.0,0
11967,главный жд вокзал,"Москва, площадь Киевского Вокзала, 1",Киевский вокзал; Kiyevsky Rail Terminal; Кіеўс...,Железнодорожный вокзал,1095876672,None,Общий обзор отзывов: Киевский вокзал оценивает...,1.0,0
16200,фспп ур база данных проверить задолженность по...,"Краснодарский край, Сочи, жилой район Лазаревс...",Лазаревский районный отдел службы судебных при...,Судебные приставы,1313648395,None,Организация занимается исполнением судебных ре...,1.0,0
16999,хендай,"Республика Адыгея (Адыгея), аул Тахтамукай, Кр...",Юг-Авто Hyundai Яблоновский; Yug-Avto Hyundai;...,Автосалон,1177113559,Автосалон и автотехцентр «Юг-Авто Hyundai Ябло...,Организация занимается продажей автомобилей с ...,1.0,0
13819,где перекусить в выборге недорого,"Ленинградская область, Выборг, улица Гагарина, 4",Буфет СССР; Bufet SSSR; СССР; Bufet Sssr,Кафе,97907735549,None,Организация «Буфет СССР» предлагает блюда в ст...,1.0,0
31026,фастфуд,"Новосибирск, Кубовая улица, 94/1",Суши Панда,Быстрое питание,124847088093,«Суши Панда» предлагает широкий ассортимент су...,Организация занимается приготовлением и достав...,1.0,0
30889,завод вторчермет,"Пермь, Соликамская улица, 283",Вторчермет НЛМК Пермь; Uvm; Вторчермет Лнмк Пе...,Приём и скупка вторсырья,1467650517,None,Организация занимается приёмом и скупкой вторс...,1.0,0
11720,центр технической инвентаризации,"Самара, улица 22-го Партсъезда, 45",Поволжское региональное БТИ; Региональное БТИ;...,БТИ,150178058866,None,Организация занимается предоставлением услуг Б...,1.0,0


## Добавляем Chain-of-Thoghts (разрешаем модели размышлять)

In [ ]:
print(f"🧪 Тестируем Smart Baseline на {len(golden_set)} сложных примерах...")

# 1. Улучшенная функция парсинга (умеет доставать JSON из текста с рассуждениями)
def parse_cot_response(content: str) -> dict:
    """
    Ищет JSON-объект в ответе модели.
    Возвращает словарь с reasoning и relevant или дефолтный Fail.
    """
    try:
        # Убираем маркеры кода, если есть
        content = re.sub(r"```json", "", content)
        content = re.sub(r"```", "", content)

        # Ищем фигурные скобки
        matches = re.findall(r"\{.*?\}", content, flags=re.DOTALL)
        if matches:
            # Берем последний найденный JSON (часто модель сначала пишет пример, потом ответ)
            last_match = matches[-1]
            data = json.loads(last_match)
            return data
    except Exception as e:
        pass

    return {"reasoning": "Parse Error", "relevant": 0}

# 2. Промпт с Chain-of-Thought (CoT)
def predict_with_cot(row, model, tokenizer):
    prompt = f"""
Ты — аналитик данных. Твоя задача — проверить, соответствует ли найденная организация запросу пользователя.

Запрос пользователя: "{row['Text']}"
Кандидат (Организация):
- Название: "{row['name']}"
- Адрес: "{row['address']}"
- Категория: "{row['normalized_main_rubric_name_ru']}"

ИНСТРУКЦИЯ:
1. Сначала напиши рассуждение (reasoning). Проанализируй совпадение по смыслу, адресу и категории. Обрати внимание на детали (например, "опт" vs "розница", "ремонт" vs "продажа").
2. Сделай итоговый вывод (verdict).
3. Верни ответ СТРОГО в формате JSON.

Формат ответа:
{{
    "reasoning": "Твой пошаговый анализ здесь...",
    "relevant": 1 (если подходит) или 0 (если нет)
}}

Не пиши ничего лишнего за пределами JSON.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,  # Увеличили токенов, так как нужно reasoning
            temperature=0.0,     # Оставляем 0 для стабильности, но рассуждение добавит гибкости
            do_sample=False
        )

    generated_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return parse_cot_response(generated_text), generated_text

🧪 Тестируем Smart Baseline на 20 сложных примерах...


In [ ]:
# 3. Запуск эксперимента
results_cot = []
fixed_count = 0

print("Запуск CoT (думаем перед ответом)...")

for idx, row in tqdm(golden_set.iterrows(), total=len(golden_set)):
    prediction_data, raw_text = predict_with_cot(row, model, tokenizer)

    pred_val = float(prediction_data.get("relevant", 0))
    true_val = float(row["relevance"])

    is_correct = (pred_val == true_val)
    if is_correct:
        fixed_count += 1

    results_cot.append({
        "query": row["Text"],
        "org_name": row["name"],
        "true_relevance": true_val,
        "pred_relevance": pred_val,
        "reasoning": prediction_data.get("reasoning", ""),
        "is_fixed": is_correct
    })

# 4. Анализ результатов
print(f"Итоги Smart Baseline (CoT):")
print(f"Было ошибок: {len(golden_set)}")
print(f"Исправлено благодаря CoT: {fixed_count}")
print(f"Осталось сложных ошибок: {len(golden_set) - fixed_count}")
print(f"Прирост точности на сложных примерах: {fixed_count / len(golden_set):.2%}")

# Показываем примеры исправлений
df_res = pd.DataFrame(results_cot)
if fixed_count > 0:
    print("Пример успешного исправления:")
    row_fixed = df_res[df_res["is_fixed"] == True].iloc[0]
    print(f"Запрос: {row_fixed['query']}")
    print(f"Орг: {row_fixed['org_name']}")
    print(f"Reasoning: {row_fixed['reasoning']}")
else:
    print("CoT не помог. Значит, проблема действительно в нехватке информации!")


🚀 Запуск CoT (думаем перед ответом)...


100%|██████████| 20/20 [14:27<00:00, 43.40s/it]


📊 Итоги Smart Baseline (CoT):
Было ошибок: 20
Исправлено благодаря CoT: 12
Осталось сложных ошибок: 8
Прирост точности на сложных примерах: 60.00%

✅ Пример успешного исправления:
Запрос: недорогие кафе геленджика
Орг: Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума
Reasoning: По запросу 'недорогие кафе геленджика' ищется организация, которая относится к категории 'кафе' и находится в городе Геленджик. Найденная организация 'Весёлая Кума' соответствует категории 'Кафе' и находится в Геленджике. Однако, информация о том, что это недорогое место, отсутствует. Кроме того, название организации может быть не совсем точным, так как в запросе нет слова 'весёлая', но это может быть просто частью названия. Адрес также соответствует городу Геленджик.


## Web-search

In [ ]:
!pip install tavily-python trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 35.5 MB/s eta 0:00:00


In [ ]:
import trafilatura
from tavily import TavilyClient
from google.colab import userdata


print("🌐 Подключаем Web-search...")

TAVILY_API_KEY = userdata.get('Tavily')
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def clean_search_query(row):
    # 1. Берем только первое название (до точки с запятой)
    name = row['name'].split(';')[0].strip()

    # 2. Пытаемся вытащить город из адреса
    # Обычно адрес это "Город, улица, дом"
    address_parts = row['address'].split(',')
    city = address_parts[0].strip()

    # Если первое слово "Россия", берем второе (это город)
    if "Россия" in city and len(address_parts) > 1:
        city = address_parts[1].strip()

    # Формируем человеческий запрос
    return f"{name} {city} официальный сайт услуги"

def search_tavily_safe(query, max_results=2):
    try:
        # search_depth="advanced" дает более качественные результаты
        response = tavily_client.search(query, search_depth="advanced", max_results=max_results)
        # Tavily возвращает сразу список словарей [{'url': ..., 'content': ...}]
        # Мы пока возьмем только URL, чтобы сохранить логику пайплайна (свой скрапер)
        # Но вообще Tavily уже возвращает распаршенный контент!
        return [r['url'] for r in response['results']]
    except Exception as e:
        print(f"Tavily Error: {e}")
        return []

# 4. Скрапер (Trafilatura)
def smart_scrape(url):
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded is None:
            return None
        text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
        return text
    except Exception:
        return None

🌐 Подключаем Web-search...
Введите ваш Tavily API Key: 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


In [ ]:
# 4. Тест "Глаз" на сложных примерах
print(f"🕵️ Проверяем, поможет ли поиск для {len(golden_set)} сложных случаев...")

web_results = []

# Берем первые 5 примеров из Golden Set для теста (чтобы не ждать долго)
# В реальном запуске уберем .head()
for idx, row in tqdm(golden_set.head(5).iterrows(), total=5):

    # Формируем поисковый запрос: Название + Город + Адрес
    # Это важно! Просто названия часто недостаточно.
    human_query = clean_search_query(row)

    # Ищем
    urls = search_tavily_safe(human_query)

    found_text = "Ничего не найдено"
    used_url = "None"

    # 2. Читаем первый доступный сайт
    for url in urls:
        text = smart_scrape(url)
        if text and len(text) > 100: # Если текста достаточно много
            found_text = text[:1000] + "..." # Берем первые 1000 символов для превью
            used_url = url
            break

        time.sleep(1)

    web_results.append({
        "query": row['Text'],
        "org_name": row['name'],
        "search_query": human_query,
        "found_url": used_url,
        "scraped_content": found_text
    })
    time.sleep(0.5)

# 5. Вывод результатов
df_web = pd.DataFrame(web_results)

# Настройка отображения, чтобы текст не обрезался
pd.set_option('display.max_colwidth', 150)
display(df_web)

🕵️ Проверяем, поможет ли поиск для 20 сложных случаев...


 80%|████████  | 4/5 [00:29<00:07,  7.07s/it]ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://gosuslugi41.ru/pgu/offices/info.htm?id=12297@egOffice
ERROR:trafilatura.downloads:download error: https://pgu.krasnodar.ru/ HTTPSConnectionPool(host='pgu.krasnodar.ru', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f9985dbf680>, 'Connection to pgu.krasnodar.ru timed out. (connect timeout=30)'))
100%|██████████| 5/5 [01:09<00:00, 13.81s/it]


,query,org_name,search_query,found_url,scraped_content
0,недорогие кафе геленджика,Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума,Весёлая Кума Краснодарский край официальный сайт услуги,https://kuma.pirnnov.ru/,"Ресторан ""Веселая Кума""\nРусская кухня\nКубанская кухня так близка русскому человеку. Драники с беконом, вареники, борщ с пампушками и фирменные н..."
1,билет надо казахстан москва,Трансмост-тур; Transmost-tour; Transmost-tur,Трансмост-тур Москва официальный сайт услуги,https://transmost-tour.ru/tourizm/,Туристическая компания «Трансмост-тур» является опытным агентством с обширной сетью офисов по Москве. Профессионализм и опыт наших сотрудников поз...
2,слесарный разметочный инструмент,Milwaukee,Milwaukee Москва официальный сайт услуги,https://milwrussia.ru/,"Для смены лезвия, требуется доработка карандаша. При извлечение лезвия удерживающий штифт ломается, либо сгибается, приходится нарезать резьбу, ст..."
3,главный жд вокзал,Киевский вокзал; Kiyevsky Rail Terminal; Кіеўскі вакзал; Kiewer Bahnhof; Gare de Kiev; Kiev Garı; Київський вокзал; Kievsky Rail Terminal; Брянски...,Киевский вокзал Москва официальный сайт услуги,https://www.msk-guide.ru/oficialnyj-sajt-i-telefon-kievskogo-vokzala.htm,Официальный сайт и телефон Киевского вокзала\nТелефон\nМногоканальный телефон единой справочной ОАО РЖД 8 (800) 775-00-00 (бесплатно с территории ...
4,фспп ур база данных проверить задолженность по фамилии,Лазаревский районный отдел службы судебных приставов; Federal Bailiffs Service; FBS; Судебные приставы; Лазаревский РОСП; Лазаревский Районный Отд...,Лазаревский районный отдел службы судебных приставов Краснодарский край официальный сайт услуги,None,Ничего не найдено
